In [ ]:
# Download data locally
import xarray as xr
import pandas as pd

from dotenv import load_dotenv, find_dotenv

import re
import os

In [ ]:
load_dotenv(find_dotenv("october_gs.env", raise_error_if_not_found=True), override=True)

EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

# Climate Data
TAS_FORECAST_ADJ = os.environ["POREALLAS_TAS_FORECAST_URI"]
ERA5_ADJ = os.environ["POREALLAS_ERA5_URI"]
TAS_FORECAST_RAW = os.environ["POREALLAS_PARSED_FORECAST_URI"]

In [ ]:
RUN_MONTH = "2026-10"
N_LEADS = 6

_months = pd.period_range(pd.Period(RUN_MONTH, "M"), periods=N_LEADS, freq="M")
FC_MONTHS = [p.month for p in _months]
FC_PERIOD = slice(_months[0].start_time.strftime("%Y-%m-%d"),
                  _months[-1].end_time.strftime("%Y-%m-%d"))

# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")

In [ ]:
def retrieve_version(uri):
    m = re.search(r"/v(\d{8})/", uri)
    return  f"v{m.group(1)}"

In [ ]:
uri = TAS_FORECAST_RAW
version = retrieve_version(uri)

In [ ]:
uri = TAS_FORECAST_RAW
version = retrieve_version(uri)
_ds = xr.open_dataset(uri, engine='zarr', chunks={'latitude': 30, 'longitude': 30, 'time': -1}, backend_kwargs={"storage_options": {"token": "anon"}})

In [ ]:
# Break out forecast into hindcast climatology and ensemble forecast
# Forecast
_ds_fc = _ds.sel(time = FC_PERIOD)
_ds_fc.to_zarr(f"{version}_parsed_forecast.zarr")



In [ ]:
# Hindcast
_ds_hc = _ds.sel(time = BASELINE_PERIOD).mean(dim = 'number')
_ds_hc.to_zarr(f"{version}_parsed_hindcast.zarr")

In [ ]:
uri = EFFECTS_URI
version = retrieve_version(uri)
_ds = xr.open_datatree(uri, engine='zarr', chunks={}, backend_kwargs={"storage_options": {"token": "anon"}})
_ds.to_zarr(f"{version}_effects.zarr")